# 📊 Valores Atípicos en Datos Aseguradores
## Diplomado en Machine Learning para Seguros — Módulo 2 · Tema 3

---

### Contenido del Notebook

| Sección | Tema | Código |
|---------|------|--------|
| **Tema 3** | Estrategias de manejo | 3.1 Eliminacion 3.2 Winzorización 3.3 Transfomaciones |
--
# 🔢 TEMA 3: Métodos Estrategias de Manejo de Outliers

Cada sección explica qué hace el programa y presenta un ejemplo integral de aplicación en un flujo de datos de siniestros.

## 3.1. Eliminación de datos sucios y outliers
Este bloque simula un dataset de siniestros con valores problemáticos y muestra cómo limpiar datos no verificados y montos negativos.

In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)
datos = pd.DataFrame({
    'id_siniestro': np.arange(1, 100),
    'monto': np.concatenate([
        np.random.normal(30, 10, 95),
        [200, 250, -50, 1000000]
    ]),
    'tipo': ['Normal']*95 + ['Error', 'Error', 'Dato Inválido', 'Catástrofe'],
    'verificado': [True]*95 + [False, False, True, False]
})

datos['monto'] = datos['monto'].round(2)

print('ANTES DE LIMPIEZA:')
print(datos.tail(10))
print(f"Media: ${datos['monto'].mean():.2f}")
print(f"Mediana: ${datos['monto'].median():.2f}")

# Estrategias de limpieza
filtro_verificados = datos['verificado'] == True
filtro_positivos = datos['monto'] > 0

datos_limpios1 = datos.loc[filtro_verificados].copy()
datos_limpios2 = datos.loc[filtro_positivos].copy()
datos_limpios3 = datos.loc[filtro_verificados & filtro_positivos].copy()

print('DESPUÉS DE LIMPIEZA:')
print(f"Datos originales: {len(datos)}")
print(f"Después quitar no verificados: {len(datos_limpios1)} (eliminados: {len(datos) - len(datos_limpios1)})")
print(f"Después quitar negativos: {len(datos_limpios2)} (eliminados: {len(datos) - len(datos_limpios2)})")
print(f"Después ambos criterios: {len(datos_limpios3)} (eliminados: {len(datos) - len(datos_limpios3)})")
print(f"Media original: ${datos['monto'].mean():.2f}")
print(f"Media limpia: ${datos_limpios3['monto'].mean():.2f}")
print(f"Diferencia: {(datos['monto'].mean() - datos_limpios3['monto'].mean()) / datos['monto'].mean() * 100:.1f}%")

ANTES DE LIMPIEZA:
    id_siniestro       monto           tipo  verificado
89            90       35.13         Normal        True
90            91       30.97         Normal        True
91            92       39.69         Normal        True
92            93       22.98         Normal        True
93            94       26.72         Normal        True
94            95       26.08         Normal        True
95            96      200.00          Error       False
96            97      250.00          Error       False
97            98      -50.00  Dato Inválido        True
98            99  1000000.00     Catástrofe       False
Media: $10132.90
Mediana: $28.84
DESPUÉS DE LIMPIEZA:
Datos originales: 99
Después quitar no verificados: 96 (eliminados: 3)
Después quitar negativos: 98 (eliminados: 1)
Después ambos criterios: 95 (eliminados: 4)
Media original: $10132.90
Media limpia: $29.03
Diferencia: 99.7%


### ¿Qué hace este programa?
1. Construye un dataset con 95 siniestros normales y 4 casos problemáticos.
2. Muestra la media y mediana antes de la limpieza.
3. Aplica tres criterios de eliminación: verificar solo casos confirmados, eliminar montos negativos y aplicar ambas reglas.
4. Compara el tamaño del dataset y el cambio de promedio tras la limpieza.

## 3.2. Winsorización
Este bloque transforma los valores extremos manteniéndolos dentro de los percentiles 5 y 95, en lugar de eliminarlos.

In [3]:
def winsorizar(datos, percentil_bajo=5, percentil_alto=95):
    limite_bajo = np.percentile(datos, percentil_bajo)
    limite_alto = np.percentile(datos, percentil_alto)
    datos_winsorizado = np.clip(datos, limite_bajo, limite_alto)
    return {
        'original': datos,
        'winsorizado': datos_winsorizado,
        'limite_bajo': limite_bajo,
        'limite_alto': limite_alto
    }

np.random.seed(42)
siniestros = np.concatenate([
    np.random.normal(30, 8, 95),
    np.array([150, 180, 200, 250, 300])
])

resultado = winsorizar(siniestros, percentil_bajo=5, percentil_alto=95)

print('=== COMPARACIÓN: ORIGINAL vs WINSORIZADO ===')
print()
print(f"Límite inferior (P5): ${resultado['limite_bajo']:.2f}")
print(f"Límite superior (P95): ${resultado['limite_alto']:.2f}")
print(f"{'Estadística':<20} {'Original':<15} {'Winsorizado':<15} {'Cambio %':<10}")
print('-' * 60)

media_orig = np.mean(resultado['original'])
media_wins = np.mean(resultado['winsorizado'])
mediana_orig = np.median(resultado['original'])
mediana_wins = np.median(resultado['winsorizado'])
std_orig = np.std(resultado['original'])
std_wins = np.std(resultado['winsorizado'])
max_orig = np.max(resultado['original'])
max_wins = np.max(resultado['winsorizado'])

print(f"{'Media':<20} ${media_orig:>13.2f} ${media_wins:>13.2f} {(media_wins - media_orig) / media_orig * 100:>8.1f}%")
print(f"{'Mediana':<20} ${mediana_orig:>13.2f} ${mediana_wins:>13.2f} {(mediana_wins - mediana_orig) / mediana_orig * 100:>8.1f}%")
print(f"{'Desv. Estándar':<20} ${std_orig:>13.2f} ${std_wins:>13.2f} {(std_wins - std_orig) / std_orig * 100:>8.1f}%")
print(f"{'Máximo':<20} ${max_orig:>13.2f} ${max_wins:>13.2f} {(max_wins - max_orig) / max_orig * 100:>8.1f}%")
print()
print(f"{'Reserva Original':<20} ${media_orig * len(resultado['original']):>13.2f}")
print(f"{'Reserva Winsorizada':<20} ${media_wins * len(resultado['winsorizado']):>13.2f}")
print(f"{'Diferencia':<20} ${(media_orig - media_wins) * len(resultado['original']):>13.2f}")

=== COMPARACIÓN: ORIGINAL vs WINSORIZADO ===

Límite inferior (P5): $16.19
Límite superior (P95): $50.08
Estadística          Original        Winsorizado     Cambio %  
------------------------------------------------------------
Media                $        38.56 $        30.39    -21.2%
Mediana              $        29.57 $        29.57      0.0%
Desv. Estándar       $        43.00 $         8.20    -80.9%
Máximo               $       300.00 $        50.08    -83.3%

Reserva Original     $      3856.01
Reserva Winsorizada  $      3039.27
Diferencia           $       816.74


### ¿Qué hace este programa?
1. Define una función que reemplaza los valores extremos por los límites P5 y P95.
2. Genera una muestra con valores normales y outliers altos.
3. Calcula medias, medianas, desviaciones, máximos y reservas antes y después de winsorizar.
4. Permite ver cómo se reduce la influencia de los extremos sin eliminar filas.

## 3.3. Transformaciones de variables
Este bloque muestra tres transformaciones comunes: arcsin-sqrt para proporciones, raíz cúbica para cambios con signos mixtos y arcsinh para datos con negativos y ceros.

In [10]:
# Transformación arcsin-sqrt para proporciones
IS = np.array([0.35, 0.42, 0.68, 0.91, 0.12, 0.55])
IS_arcsin = np.arcsin(np.sqrt(IS))
IS_original = np.sin(IS_arcsin) ** 2

print('Transformación arcsin-sqrt:')
print('Original:', np.round(IS, 4))
print('Transformada:', np.round(IS_arcsin, 4))
print('Recuperada:', np.round(IS_original, 4))

# Transformación raíz cúbica para cambios con signo
delta_reserva = np.array([-500000, -20000, 0, 15000, 80000, 3000000])
delta_cbrt = np.cbrt(delta_reserva)

print('Transformación raíz cúbica:')
print('Delta reserva:', delta_reserva)
print('Cbrt:', np.round(delta_cbrt, 4))

# Transformación arcsinh para datos con ceros y negativos
resultado = np.array([-500000, -20000, 0, 15000, 80000, 3000000])
resultado_asinh = np.arcsinh(resultado)
resultado_recuperado = np.sinh(resultado_asinh)

print('Transformación arcsinh:')
print('Original:', resultado)
print('Asinh:', np.round(resultado_asinh, 4))
print('Recuperado:', np.round(resultado_recuperado, 4))

Transformación arcsin-sqrt:
Original: [0.35 0.42 0.68 0.91 0.12 0.55]
Transformada: [0.6331 0.7051 0.9695 1.2661 0.3537 0.8355]
Recuperada: [0.35 0.42 0.68 0.91 0.12 0.55]
Transformación raíz cúbica:
Delta reserva: [-500000  -20000       0   15000   80000 3000000]
Cbrt: [-79.3701 -27.1442   0.      24.6621  43.0887 144.225 ]
Transformación arcsinh:
Original: [-500000  -20000       0   15000   80000 3000000]
Asinh: [-13.8155 -10.5966   0.      10.309   11.9829  15.6073]
Recuperado: [-500000.  -20000.       0.   15000.   80000. 3000000.]


### ¿Qué hace este programa?
1. Aplica la transformada arcsin-sqrt a tasas de siniestralidad para estabilizar varianzas en proporciones.
2. Usa raíz cúbica para transformar cambios de reserva que pueden ser negativos o positivos.
3. Demuestra arcsinh como alternativa robusta para datos con valores negativos y ceros.
4. Recupera los valores originales usando la función inversa cuando la transformación es reversible.

## 4. Ejemplo integral
Aquí combinamos los tres enfoques en un flujo único: limpiar los datos, winsorizar los montos extremos y aplicar transformaciones a los cambios de reserva.

In [9]:
# Reusar el dataset limpio de la primera sección
datos_limpios = datos[(datos['verificado'] == True) & (datos['monto'] > 0)].copy()
montos = datos_limpios['monto'].to_numpy()

# Winsorizar montos limítrofes
resultado_wins = winsorizar(montos, percentil_bajo=5, percentil_alto=95)
datos_limpios['monto_wins'] = resultado_wins['winsorizado']

# Transformación de delta de reserva usando cbrt y arcsinh
datos_limpios['delta_reserva'] = datos_limpios['monto_wins'] - np.median(datos_limpios['monto_wins'])
datos_limpios['delta_cbrt'] = np.cbrt(datos_limpios['delta_reserva'])
datos_limpios['delta_asinh'] = np.arcsinh(datos_limpios['delta_reserva'])

print('Resumen integral: limpieza + winsorización + transformaciones')
print('-' * 70)
print(f"Montos originales: {len(montos)} filas")
print(f"Monto mínimo original: ${montos.min():.2f}")
print(f"Monto máximo original: ${montos.max():.2f}")
print(f"Monto máximo winsorizado: ${datos_limpios['monto_wins'].max():.2f}")
print('Estadísticas originales vs winsorizadas:')
print(datos_limpios[['monto', 'monto_wins']].describe().round(2))

print('Primeras filas del dataset transformado:')
print(datos_limpios[['id_siniestro', 'monto', 'monto_wins', 'delta_reserva', 'delta_cbrt', 'delta_asinh']].head())

Resumen integral: limpieza + winsorización + transformaciones
----------------------------------------------------------------------
Montos originales: 95 filas
Monto mínimo original: $3.80
Monto máximo original: $48.52
Monto máximo winsorizado: $44.91
Estadísticas originales vs winsorizadas:
       monto  monto_wins
count  95.00       95.00
mean   29.03       29.13
std     9.20        8.71
min     3.80       12.64
25%    23.98       23.98
50%    28.62       28.62
75%    35.05       35.05
max    48.52       44.92
Primeras filas del dataset transformado:
   id_siniestro  monto  monto_wins  delta_reserva  delta_cbrt  delta_asinh
0             1  34.97      34.970          6.350    1.851788     2.547745
1             2  28.62      28.620          0.000    0.000000     0.000000
2             3  36.48      36.480          7.860    1.988265     2.758956
3             4  45.23      44.915         16.295    2.535234     3.484946
4             5  27.66      27.660         -0.960   -0.986485    

### ¿Qué muestra el ejemplo integral?
1. Empieza con el mismo dataset de siniestros y elimina valores no verificados o negativos.
2. Winsoriza los montos para reducir el impacto de extremos sin perder filas.
3. Calcula diferencias de reserva relativas a la mediana y aplica transformaciones suaves.
4. Genera estadísticas comparativas y presenta las primeras filas del resultado final.